# Multi-Subject NF fMRI — Yeo-17 Functional Connectivity Analysis
Parcel-level segregation, integration, normalized segregation, and participation coefficient across subjects, sessions, and runs.

## Imports

In [ ]:
import os
import glob
import pickle
import re
from collections import defaultdict
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def load_yeo_800(yeo=17):
    import os, pickle, pandas as pd, numpy as np
    if yeo != 17:
        raise NotImplementedError("Only Yeo-17 is implemented for 800 parcels.")
    pickle_file = "../../../Data/rsfMRI_utils.pkl"
    with open(pickle_file, "rb") as f:
        rsfMRI_utils = pickle.load(f)
    csv_path   = "../utils/matched_labels_exact_800.csv"
    df_mapping = pd.read_csv(csv_path)
    labels     = df_mapping["label_df2"].iloc[1:].reset_index(drop=True)
    networks_from_csv = np.array([r.split("_")[2] for r in labels])
    yeo_order_17 = [
        "VisCent", "VisPeri", "SomMotA", "SomMotB",
        "DorsAttnA", "DorsAttnB", "SalVentAttnA", "SalVentAttnB",
        "LimbicB", "LimbicA", "ContA", "ContB", "ContC",
        "DefaultA", "DefaultB", "DefaultC", "TempPar",
    ]
    name_to_id = {name: idx for idx, name in enumerate(yeo_order_17)}
    yeoROIs    = np.array([name_to_id[n] for n in networks_from_csv], dtype=int)
    yeoOrder   = np.arange(len(yeoROIs))
    yeo_net    = name_to_id
    return yeoOrder, yeoROIs, yeo_net, rsfMRI_utils

## Configuration
Only this cell needs editing when you move to a different machine or dataset.

In [ ]:
f=open('../../../Data/rsfMRI_utils.pkl','rb'); d=pickle.load(f); print(d.keys()); print({k: list(v.keys()) if hasattr(v,'keys') else type(v) for k,v in d.items()})

In [ ]:
from nilearn import datasets
import pandas as pd

# Fetch Schaefer 800-parcel, 17-network atlas
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=800, yeo_networks=17, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else l for l in atlas.labels]

# labels look like: '17Networks_LH_VisCent_1', '17Networks_LH_VisPeri_2', ...
df = pd.DataFrame({'label_df2': labels})
df.to_csv('../utils/matched_labels_exact_800.csv', index=False)
print(df.head(10))
print(f"Total parcels: {len(df)}")

In [ ]:
import pandas as pd
df = pd.read_csv('../utils/matched_labels_exact_800.csv')
print(df.columns.tolist())
print(df.head(5))

In [ ]:
# Root folder — one sub-directory per subject, e.g. data/NF/002GNV/
DATA_ROOT = "../../../Data/"

YEO17_NAMES = [
    "VisCent",
    "VisPeri",
    "SomMotA",
    "SomMotB",
    "DorsAttnA",
    "DorsAttnB",
    "SalVentAttnA",
    "SalVentAttnB",
    "LimbicA",
    "LimbicB",
    "ContA",
    "ContB",
    "ContC",
    "DefaultA",
    "DefaultB",
    "DefaultC",
    "TempPar",
]

YEO17_COLORS = [
    "#781286",  # 1  VisCent
    "#ff0101",  # 2  VisPeri
    "#4682b4",  # 3  SomMotA
    "#118ab2",  # 4  SomMotB
    "#2e8b57",  # 5  DorsAttnA
    "#00760e",  # 6  DorsAttnB
    "#dcf8a4",  # 7  SalVentAttnA
    "#c8e96e",  # 8  SalVentAttnB
    "#c9f8c9",  # 9  LimbicA
    "#e69422",  # 10 LimbicB
    "#f0e442",  # 11 ContA
    "#f0a800",  # 12 ContB
    "#f05a00",  # 13 ContC
    "#cd3d4f",  # 14 DefaultA
    "#e044af",  # 15 DefaultB
    "#ff6fc8",  # 16 DefaultC
    "#b15928",  # 17 TempPar
]

# Load Yeo-17 labels once (same for all subjects)
_, yeoROIs17, _, _ = load_yeo_800(yeo=17)
YEO17_LABELS = np.array(yeoROIs17).astype(int).ravel()  # (800,), labels 0-17

## 1. Data Loading
Auto-discover all subject folders, then load every session FC file for each subject.
The Yeo-17 atlas labels are loaded once from the shared parcellation (not per subject).

In [ ]:
def discover_subjects(data_root):
    """Return sorted list of subject IDs (= sub-folder names) in data_root."""
    return sorted(
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
    )


def load_subject(data_root, subject_id):
    """
    Load one subject's session FC data.

    Returns
    -------
    sessions : dict  {session_id -> {run_key -> {"condition": str, "FC": ndarray}}}
    """
    subj_dir = os.path.join(data_root, subject_id)
    fc_files = sorted(glob.glob(os.path.join(subj_dir, f"{subject_id}_V*_FC_Schaefer.pkl")))
    print(f"  {subject_id}: {len(fc_files)} session file(s)")
    sessions = {}
    for fc_path in fc_files:
        session_id = os.path.basename(fc_path).split("_")[1]
        with open(fc_path, "rb") as f:
            data = pickle.load(f)
        run_map = {}
        for key, fc in data["FC_FD"].items():
            if not key.endswith("_800"):
                continue
            m = re.match(r"V\d+_Run(\d+)(?:_(Transfer|NoFeedback))?_800", key)
            if m is None:
                continue
            run_key   = f"Run{m.group(1)}"
            condition = m.group(2) if m.group(2) else "Feedback"
            run_map[run_key] = {"condition": condition, "FC": fc}
        sessions[session_id] = run_map
    return sessions


# --- Discover and load all subjects ---
subject_ids = discover_subjects(DATA_ROOT)
print(f"Subjects found: {subject_ids}\n")

# all_data[subject_id] = {"sessions": ..., "yeo_labels": ...}
all_data = {}
for sid in subject_ids:
    sessions = load_subject(DATA_ROOT, sid)
    all_data[sid] = {"sessions": sessions, "yeo_labels": YEO17_LABELS}

print(f"\nLoaded {len(all_data)} subject(s).")

In [ ]:
count = 0
subjects_with_run = []

for sid, subj_data in all_data.items():
    sessions = subj_data["sessions"]
    
    if "V03" in sessions and "Run8" in sessions["V03"]:
        count += 1
        subjects_with_run.append(sid)

print("Subjects with V03 Run8:", count)
print(subjects_with_run)

In [ ]:
for sid, subj_data in all_data.items():
    sessions = subj_data["sessions"]
    
    if "V03" in sessions and "Run8" in sessions["V03"]:
        del sessions["V03"]["Run8"]

In [ ]:
counts = defaultdict(int)

for sid, subj_data in all_data.items():
    for sess, runs in subj_data["sessions"].items():
        for run in runs.keys():
            counts[(sess, run)] += 1

rows = []
for (sess, run), n in counts.items():
    rows.append({"session": sess, "run": run, "n_subjects": n})

df_counts = pd.DataFrame(rows)
table = df_counts.pivot(index="session", columns="run", values="n_subjects")

print(table)

## 2. Metric Computation — Parcel-Level First
For each subject and session:
1. Fisher z-transform the raw parcel FC matrix.
2. Restrict to **cortical parcels (Yeo labels 1–17)** → ≈200×200 matrix.
3. For each network k:
   - **Segregation**: mean FC among all within-network parcel pairs (diagonal excluded).
   - **Integration**: mean FC between parcels in network k and parcels in all other networks.
   - **Normalized segregation**: `(seg − int) / (seg + int)`.
4. **Participation coefficient** (parcel level): proportion of each parcel's total connectivity that goes to other networks.

In [ ]:
def compute_yeo17_metrics(fc, yeo_labels):
    """
    Parcel-level Yeo-17 segregation / integration / participation coefficient.

    Parameters
    ----------
    fc         : (N, N) array  raw parcel FC (Pearson r)
    yeo_labels : (N,)   int array  Yeo-17 labels 0–17 in same parcel order

    Returns
    -------
    segregation            : (17,) array
    integration            : (17,) array
    normalized_segregation : (17,) array
    pc_parcel              : (~800,) array  participation coefficient per cortical parcel
    """
    # Step 1 – restrict to cortical parcels (Yeo 1-17)
    cortical_mask = (yeo_labels >= 1) & (yeo_labels <= 17)
    fc_cx  = np.array(fc[np.ix_(cortical_mask, cortical_mask)], dtype=float, copy=True)
    yeo_cx = yeo_labels[cortical_mask].astype(int)

    # Step 2 – Fisher z-transform, remove self-connections
    fc_cx = np.clip(fc_cx, -0.999999, 0.999999)
    fc_z  = np.arctanh(fc_cx)
    np.fill_diagonal(fc_z, np.nan)

    # Step 3 – per-network seg/int
    K = 17
    segregation = np.full(K, np.nan)
    integration = np.full(K, np.nan)

    for k in range(1, K + 1):
        mask_k     = (yeo_cx == k)
        mask_other = (yeo_cx != k)

        if mask_k.sum() >= 2:
            within = fc_z[np.ix_(mask_k, mask_k)].copy()
            np.fill_diagonal(within, np.nan)
            segregation[k - 1] = np.nanmean(within)

        if mask_k.sum() > 0 and mask_other.sum() > 0:
            integration[k - 1] = np.nanmean(fc_z[np.ix_(mask_k, mask_other)])

    normalized_segregation = (segregation - integration) / (segregation + integration)

    # Step 4 – participation coefficient (positive weights only, matching utils/metrics.py)
    fc_pos = np.where(fc_z > 0, fc_z, 0.0)
    fc_pos = np.nan_to_num(fc_pos, nan=0.0)
    np.fill_diagonal(fc_pos, 0.0)

    N = fc_pos.shape[0]
    pc = np.full(N, np.nan)
    for i in range(N):
        k_i = np.sum(fc_pos[i, :])
        if k_i <= 0:
            continue
        sum_sq = sum((np.sum(fc_pos[i, yeo_cx == s]) / k_i) ** 2 for s in range(1, 18))
        pc[i] = 1.0 - sum_sq
    pc = np.clip(pc, 0.0, 1.0)

    return segregation, integration, normalized_segregation, pc


def compute_all_metrics(all_data):
    """Add segregation / integration / normalized_segregation / pc_parcel to every run entry in-place."""
    for subj in all_data.values():
        yeo = subj["yeo_labels"]
        for sess_runs in subj["sessions"].values():
            for rd in sess_runs.values():
                seg, intg, normseg, pc_parcel = compute_yeo17_metrics(rd["FC"], yeo)
                rd["segregation"]            = seg
                rd["integration"]            = intg
                rd["normalized_segregation"] = normseg
                rd["pc_parcel"]              = pc_parcel


compute_all_metrics(all_data)
print("Metrics computed for all subjects.")

## 11. Export
Save every run-level metric to a CSV for downstream analysis.

In [ ]:
import pandas as pd

export_rows = []
for sid, subj in all_data.items():
    yeo = subj["yeo_labels"]
    cortical_mask = (yeo >= 1) & (yeo <= 17)
    yeo_cx = yeo[cortical_mask].astype(int)          # (n_cortical,)  labels 1-17
    cortical_indices = np.where(cortical_mask)[0]    # original parcel indices

    for sess_id, sess_runs in subj["sessions"].items():
        for run_key in sorted(sess_runs.keys(), key=lambda x: int(x.replace("Run", ""))):
            rd = sess_runs[run_key]

            # ── parcel-level rows (one per cortical parcel) ──────────────────
            pc = rd["pc_parcel"]   # (~800,) already cortical-only from compute_yeo17_metrics
            for i, (orig_idx, yeo_lbl) in enumerate(zip(cortical_indices, yeo_cx)):
                # network-level metrics repeated on each parcel row for convenience
                net_idx = yeo_lbl - 1
                export_rows.append({
                    "subject":                sid,
                    "session":                sess_id,
                    "run":                    run_key,
                    "condition":              rd["condition"],
                    "parcel":                 int(orig_idx),
                    "yeo_label":              int(yeo_lbl),
                    "pc_parcel":              float(pc[i]),
                    "network":                YEO17_NAMES[net_idx],
                    "segregation":            rd["segregation"][net_idx],
                    "integration":            rd["integration"][net_idx],
                    "normalized_segregation": rd["normalized_segregation"][net_idx],
                })

df_export = pd.DataFrame(export_rows)
out_path = os.path.join(DATA_ROOT, "all_subjects_yeo17_metrics.csv")
df_export.to_csv(out_path, index=False)
print(f"Saved {len(df_export)} rows to {out_path}")
print(f"Columns: {df_export.columns.tolist()}")
df_export.head(5)

In [ ]:
# Inspect one run's keys
sid = next(iter(all_data))
sess = next(iter(all_data[sid]["sessions"]))
run  = next(iter(all_data[sid]["sessions"][sess]))
rd   = all_data[sid]["sessions"][sess][run]
print("Keys:", list(rd.keys()))
print("yeo_labels:", all_data[sid].get("yeo_labels", "MISSING")[:5] if "yeo_labels" in all_data[sid] else "MISSING")